In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red"> 머닝러신 실제모델링</span>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.feature_selection import SelectKBest, f_regression
import xgboost as xgb
import joblib
warnings.filterwarnings('ignore')

# 한글 폰트 설정
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
except:
    try:
        plt.rcParams['font.family'] = 'AppleGothic'
    except:
        plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 파스텔 색상 팔레트
pastel_colors = {
    'primary': '#FFB6C1', 'secondary': '#87CEEB', 'accent': '#DDA0DD',
    'success': '#98FB98', 'warning': '#F0E68C', 'danger': '#FFA07A',
    'info': '#AFEEEE', 'light': '#F5F5DC', 'purple': '#E6E6FA'
}

print("🚀 고급 머신러닝 수요예측 모델 시작")
print("="*70)

# 1. 데이터 로딩 및 전처리
def load_and_preprocess_data():
    """데이터 로딩 및 고급 전처리"""
    
    # 전처리된 데이터 로딩
    df = pd.read_csv('C:/ai_x2/source/proz2/외국인입국자_전처리완료_딥러닝용.csv', encoding='utf-8-sig')
    
    print(f"✅ 데이터 로딩 완료: {len(df):,}개 레코드")
    print(f"📅 기간: {df['연도'].min()}년 {df['월'].min()}월 ~ {df['연도'].max()}년 {df['월'].max()}월")
    print(f"🌍 국가 수: {df['국적'].nunique()}개")
    print(f"🎯 목적 종류: {df['목적'].nunique()}개")
    
    return df

df = load_and_preprocess_data()

# 2. 고급 특징 엔지니어링
class AdvancedFeatureEngineer:
    """고급 특징 엔지니어링 클래스"""
    
    def __init__(self):
        self.label_encoders = {}
        self.scalers = {}
        
    def create_time_features(self, df):
        """시간 관련 특징 생성"""
        df = df.copy()
        
        # 기본 시간 특징
        df['연월'] = df['연도'] * 100 + df['월']
        df['계절_코드'] = df['계절'].map({'봄': 1, '여름': 2, '가을': 3, '겨울': 4})
        
        # 순환 특징 (사인/코사인 변환)
        df['월_sin'] = np.sin(2 * np.pi * df['월'] / 12)
        df['월_cos'] = np.cos(2 * np.pi * df['월'] / 12)
        df['분기_sin'] = np.sin(2 * np.pi * df['분기'] / 4)
        df['분기_cos'] = np.cos(2 * np.pi * df['분기'] / 4)
        
        # 연도 정규화
        df['연도_정규화'] = (df['연도'] - df['연도'].min()) / (df['연도'].max() - df['연도'].min())
        
        # 트렌드 특징
        df['선형트렌드'] = df['시계열순서']
        df['제곱트렌드'] = df['시계열순서'] ** 2
        
        return df
    
    def create_lag_features(self, df):
        """래그 특징 생성 및 개선"""
        df = df.copy()
        
        # 기존 래그 특징 활용
        existing_lags = ['입국자수_1개월전', '입국자수_3개월전', '입국자수_12개월전']
        
        # 추가 래그 특징
        df_sorted = df.sort_values(['국적', '목적', '시계열순서'])
        
        for country_purpose, group in df_sorted.groupby(['국적', '목적']):
            if len(group) > 6:
                # 6개월, 24개월 래그
                group = group.copy()
                group['입국자수_6개월전'] = group['입국자수'].shift(6)
                group['입국자수_24개월전'] = group['입국자수'].shift(24)
                
                # 업데이트
                df.loc[group.index, '입국자수_6개월전'] = group['입국자수_6개월전']
                df.loc[group.index, '입국자수_24개월전'] = group['입국자수_24개월전']
        
        # 래그 특징 비율
        df['증감률_1개월'] = (df['입국자수'] - df['입국자수_1개월전']) / (df['입국자수_1개월전'] + 1)
        df['증감률_3개월'] = (df['입국자수'] - df['입국자수_3개월전']) / (df['입국자수_3개월전'] + 1)
        df['증감률_12개월'] = (df['입국자수'] - df['입국자수_12개월전']) / (df['입국자수_12개월전'] + 1)
        
        return df
    
    def create_categorical_features(self, df):
        """범주형 특징 고급 인코딩"""
        df = df.copy()
        
        # 빈도 기반 인코딩
        country_freq = df['국적'].value_counts()
        df['국적_빈도'] = df['국적'].map(country_freq)
        df['국적_로그빈도'] = np.log1p(df['국적_빈도'])
        
        purpose_freq = df['목적'].value_counts()
        df['목적_빈도'] = df['목적'].map(purpose_freq)
        
        # 타겟 기반 인코딩 (평균 입국자수)
        country_mean = df.groupby('국적')['입국자수'].mean()
        df['국적_평균입국자수'] = df['국적'].map(country_mean)
        
        purpose_mean = df.groupby('목적')['입국자수'].mean()
        df['목적_평균입국자수'] = df['목적'].map(purpose_mean)
        
        # 라벨 인코딩
        for col in ['국적', '목적', '계절']:
            if col not in self.label_encoders:
                self.label_encoders[col] = LabelEncoder()
                df[f'{col}_인코딩'] = self.label_encoders[col].fit_transform(df[col].astype(str))
            else:
                df[f'{col}_인코딩'] = self.label_encoders[col].transform(df[col].astype(str))
        
        return df
    
    def create_interaction_features(self, df):
        """상호작용 특징 생성"""
        df = df.copy()
        
        # 국적-목적 상호작용
        df['국적목적_조합'] = df['국적'].astype(str) + '_' + df['목적'].astype(str)
        combo_freq = df['국적목적_조합'].value_counts()
        df['국적목적_빈도'] = df['국적목적_조합'].map(combo_freq)
        
        # 계절-목적 상호작용
        df['계절목적_조합'] = df['계절'].astype(str) + '_' + df['목적'].astype(str)
        
        # 시간-국적 상호작용
        df['월국적_상호작용'] = df['월'] * df['국적_빈도']
        df['연도국적_상호작용'] = df['연도'] * df['국적_빈도']
        
        return df
    
    def create_statistical_features(self, df):
        """통계적 특징 생성"""
        df = df.copy()
        
        # 국적별 통계 특징
        country_stats = df.groupby('국적')['입국자수'].agg([
            'mean', 'std', 'min', 'max', 'median'
        ]).reset_index()
        country_stats.columns = ['국적', '국적_평균', '국적_표준편차', '국적_최소', '국적_최대', '국적_중앙값']
        
        df = df.merge(country_stats, on='국적', how='left')
        
        # 목적별 통계 특징
        purpose_stats = df.groupby('목적')['입국자수'].agg([
            'mean', 'std', 'min', 'max'
        ]).reset_index()
        purpose_stats.columns = ['목적', '목적_평균', '목적_표준편차', '목적_최소', '목적_최대']
        
        df = df.merge(purpose_stats, on='목적', how='left')
        
        # 계절별 통계 특징
        season_stats = df.groupby('계절')['입국자수'].agg(['mean', 'std']).reset_index()
        season_stats.columns = ['계절', '계절_평균', '계절_표준편차']
        
        df = df.merge(season_stats, on='계절', how='left')
        
        return df
    
    def fit_transform(self, df):
        """전체 특징 엔지니어링 실행"""
        print("🔧 고급 특징 엔지니어링 시작...")
        
        df = self.create_time_features(df)
        print("   ✅ 시간 특징 생성 완료")
        
        df = self.create_lag_features(df)
        print("   ✅ 래그 특징 생성 완료")
        
        df = self.create_categorical_features(df)
        print("   ✅ 범주형 특징 인코딩 완료")
        
        df = self.create_interaction_features(df)
        print("   ✅ 상호작용 특징 생성 완료")
        
        df = self.create_statistical_features(df)
        print("   ✅ 통계적 특징 생성 완료")
        
        print(f"🎯 최종 특징 수: {df.shape[1]}개")
        return df

# 특징 엔지니어링 실행
feature_engineer = AdvancedFeatureEngineer()
df_featured = feature_engineer.fit_transform(df)

# 3. 고급 모델링 클래스
class AdvancedForecastingEnsemble:
    """고급 앙상블 예측 모델"""
    
    def __init__(self):
        self.models = {
            'RandomForest': RandomForestRegressor(
                n_estimators=200, max_depth=15, min_samples_split=5,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            ),
            'XGBoost': xgb.XGBRegressor(
                n_estimators=200, max_depth=8, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
            ),
            'GradientBoosting': GradientBoostingRegressor(
                n_estimators=200, max_depth=8, learning_rate=0.1,
                subsample=0.8, random_state=42
            ),
            'Ridge': Ridge(alpha=10.0),
            'Lasso': Lasso(alpha=1.0)
        }
        self.trained_models = {}
        self.feature_importance = {}
        self.performance_metrics = {}
        self.scaler = StandardScaler()
        self.feature_selector = None
        
    def prepare_features(self, data, target_col='입국자수', feature_selection=True):
        """특징 준비 및 선택"""
        
        # 수치형 특징만 선택
        numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
        feature_cols = [col for col in numeric_cols if col != target_col]
        
        X = data[feature_cols].fillna(0)  # 결측치 처리
        y = data[target_col]
        
        # 무한대 값 처리
        X = X.replace([np.inf, -np.inf], 0)
        
        # 특징 선택 (상위 50개 특징)
        if feature_selection and len(feature_cols) > 50:
            if self.feature_selector is None:
                self.feature_selector = SelectKBest(score_func=f_regression, k=50)
                X_selected = self.feature_selector.fit_transform(X, y)
                selected_features = [feature_cols[i] for i in self.feature_selector.get_support(indices=True)]
            else:
                X_selected = self.feature_selector.transform(X)
                selected_features = [feature_cols[i] for i in self.feature_selector.get_support(indices=True)]
            
            print(f"🎯 특징 선택: {len(feature_cols)} → {len(selected_features)}개")
            return pd.DataFrame(X_selected, columns=selected_features), y, selected_features
        
        return X, y, feature_cols
    
    def train_models(self, data, target_col='입국자수', test_size=0.2):
        """모든 모델 학습 및 평가"""
        print(f"\n🤖 {len(self.models)}개 고급 모델 학습 시작...")
        
        X, y, feature_cols = self.prepare_features(data, target_col)
        
        # 시계열 분할 (최근 20% 테스트)
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        
        print(f"📊 학습 데이터: {len(X_train):,}개, 테스트 데이터: {len(X_test):,}개")
        
        # 정규화
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        for name, model in self.models.items():
            print(f"   🔄 {name} 학습 중...")
            
            try:
                # 모델별 데이터 선택
                if name in ['Ridge', 'Lasso']:
                    model.fit(X_train_scaled, y_train)
                    y_pred = model.predict(X_test_scaled)
                else:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                
                # 음수 예측값 보정
                y_pred = np.maximum(y_pred, 0)
                
                # 성능 평가
                mae = mean_absolute_error(y_test, y_pred)
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                r2 = r2_score(y_test, y_pred)
                
                # MAPE 계산 (안전하게)
                y_test_safe = np.where(y_test == 0, 1, y_test)
                mape = np.mean(np.abs((y_test - y_pred) / y_test_safe)) * 100
                mape = min(mape, 999.9)
                
                self.performance_metrics[name] = {
                    'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2, 'MAPE': mape
                }
                
                # 특징 중요도
                if hasattr(model, 'feature_importances_'):
                    importance_df = pd.DataFrame({
                        'feature': feature_cols,
                        'importance': model.feature_importances_
                    }).sort_values('importance', ascending=False)
                    self.feature_importance[name] = importance_df
                
                self.trained_models[name] = model
                print(f"   ✅ {name}: MAPE={mape:.2f}%, R²={r2:.3f}")
                
            except Exception as e:
                print(f"   ❌ {name} 학습 실패: {e}")
        
        return X_train, X_test, y_train, y_test, feature_cols
    
    def predict_future(self, future_data, feature_cols):
        """미래 예측"""
        predictions = {}
        
        for name, model in self.trained_models.items():
            try:
                # 특징 데이터 준비
                X_future = future_data[feature_cols].fillna(0)
                X_future = X_future.replace([np.inf, -np.inf], 0)
                
                # 특징 선택 적용
                if self.feature_selector is not None:
                    X_future = pd.DataFrame(
                        self.feature_selector.transform(X_future),
                        columns=[feature_cols[i] for i in self.feature_selector.get_support(indices=True)]
                    )
                
                # 예측
                if name in ['Ridge', 'Lasso']:
                    X_future_scaled = self.scaler.transform(X_future)
                    pred = model.predict(X_future_scaled)
                else:
                    pred = model.predict(X_future)
                
                # 음수 방지 및 현실적 범위 조정
                pred = np.maximum(pred, 0)
                pred = np.minimum(pred, 1000000)  # 최대 100만명
                
                predictions[name] = pred
                print(f"   ✅ {name} 예측 완료")
                
            except Exception as e:
                print(f"   ❌ {name} 예측 실패: {e}")
        
        return predictions
    
    def get_ensemble_prediction(self, predictions):
        """성능 기반 앙상블 예측"""
        if not predictions:
            return np.array([]), {}
        
        # 성능 기반 가중치
        weights = {}
        total_weight = 0
        
        for name in predictions.keys():
            if name in self.performance_metrics:
                r2_score = max(0, self.performance_metrics[name]['R²'])
                mape_score = max(1, self.performance_metrics[name]['MAPE'])
                weight = (r2_score ** 2) / (mape_score / 100)  # R² 제곱 / MAPE
                weights[name] = weight
                total_weight += weight
        
        # 가중 평균
        ensemble_pred = np.zeros_like(list(predictions.values())[0])
        
        for name, pred in predictions.items():
            if total_weight > 0:
                weight = weights.get(name, 0) / total_weight
                ensemble_pred += pred * weight
                print(f"   📊 {name} 가중치: {weight:.3f}")
        
        return ensemble_pred, weights

# 4. 미래 데이터 생성 함수
def create_future_data(df_featured, target_year=2026):
    """2026년 미래 데이터 생성"""
    print(f"\n🔮 {target_year}년 미래 데이터 생성...")
    
    # 기본 구조 생성
    future_data = []
    
    # 주요 국가들 (상위 15개)
    top_countries = df_featured.groupby('국적')['입국자수'].sum().nlargest(15).index.tolist()
    
    # 모든 목적
    purposes = df_featured['목적'].unique()
    
    # 계절 매핑
    season_map = {1: '겨울', 2: '겨울', 3: '봄', 4: '봄', 5: '봄', 
                  6: '여름', 7: '여름', 8: '여름', 9: '가을', 
                  10: '가을', 11: '가을', 12: '겨울'}
    
    for month in range(1, 13):
        for country in top_countries:
            for purpose in purposes:
                # 기본 정보
                quarter = (month - 1) // 3 + 1
                season = season_map[month]
                
                # 시계열 순서 계산 (2005년 1월부터 계산)
                time_order = (target_year - 2005) * 12 + month
                
                # 과거 데이터에서 패턴 학습
                historical = df_featured[
                    (df_featured['국적'] == country) & 
                    (df_featured['목적'] == purpose) &
                    (df_featured['월'] == month)
                ]
                
                if len(historical) > 0:
                    # 최근 3년 평균
                    recent_data = historical[historical['연도'] >= 2022]
                    if len(recent_data) > 0:
                        avg_visitors = recent_data['입국자수'].mean()
                        avg_12m_ago = recent_data['입국자수_12개월전'].mean()
                        avg_3m_avg = recent_data['입국자수_3개월평균'].mean()
                        avg_12m_avg = recent_data['입국자수_12개월평균'].mean()
                    else:
                        avg_visitors = historical['입국자수'].mean()
                        avg_12m_ago = historical['입국자수_12개월전'].mean()
                        avg_3m_avg = historical['입국자수_3개월평균'].mean()
                        avg_12m_avg = historical['입국자수_12개월평균'].mean()
                else:
                    # 기본값
                    avg_visitors = 1000
                    avg_12m_ago = 1000
                    avg_3m_avg = 1000
                    avg_12m_avg = 1000
                
                # 결측치 처리
                avg_12m_ago = avg_12m_ago if not pd.isna(avg_12m_ago) else avg_visitors
                avg_3m_avg = avg_3m_avg if not pd.isna(avg_3m_avg) else avg_visitors
                avg_12m_avg = avg_12m_avg if not pd.isna(avg_12m_avg) else avg_visitors
                
                future_record = {
                    '국적': country,
                    '목적': purpose,
                    '연도': target_year,
                    '월': month,
                    '분기': quarter,
                    '계절': season,
                    '코로나기간': 0,  # 포스트 코로나
                    '시계열순서': time_order,
                    '입국자수': avg_visitors,  # 예측할 타겟
                    '입국자수_1개월전': avg_visitors * 0.95,
                    '입국자수_3개월전': avg_visitors * 0.9,
                    '입국자수_12개월전': avg_12m_ago,
                    '입국자수_3개월평균': avg_3m_avg,
                    '입국자수_12개월평균': avg_12m_avg,
                    '전년동월대비증감률': 0.1  # 10% 성장 가정
                }
                
                future_data.append(future_record)
    
    future_df = pd.DataFrame(future_data)
    print(f"✅ {len(future_df):,}개 미래 레코드 생성")
    print(f"   📅 기간: {target_year}년 1월 ~ 12월")
    print(f"   🌍 국가: {len(top_countries)}개")
    print(f"   🎯 목적: {len(purposes)}개")
    
    return future_df

# 5. 모델 학습 실행
print("\n" + "="*70)
print("🎯 고급 머신러닝 모델 학습 시작")
print("="*70)

ensemble = AdvancedForecastingEnsemble()
X_train, X_test, y_train, y_test, feature_cols = ensemble.train_models(df_featured)

# 6. 성능 분석
def analyze_model_performance(ensemble):
    """모델 성능 상세 분석"""
    print(f"\n📊 모델 성능 상세 분석")
    print("-" * 60)
    
    performance_df = pd.DataFrame(ensemble.performance_metrics).T
    performance_df = performance_df.sort_values('MAPE')
    
    for idx, (model, metrics) in enumerate(performance_df.iterrows(), 1):
        print(f"{idx}. {model:15s} | MAPE: {metrics['MAPE']:6.2f}% | R²: {metrics['R²']:6.3f} | RMSE: {metrics['RMSE']:8.0f}")
    
    best_model = performance_df.index[0]
    print(f"\n🏆 최고 성능: {best_model} (MAPE: {performance_df.loc[best_model, 'MAPE']:.2f}%)")
    
    return best_model, performance_df

best_model, performance_df = analyze_model_performance(ensemble)

# 7. 2026년 예측 생성
print(f"\n" + "="*70)
print("🔮 2026년 수요예측 생성")
print("="*70)

# 미래 데이터 생성
future_df = create_future_data(df_featured, 2026)

# 특징 엔지니어링 적용
future_featured = feature_engineer.fit_transform(future_df)

# 예측 실행
print(f"\n🚀 예측 실행 중...")
predictions = ensemble.predict_future(future_featured, feature_cols)
ensemble_pred, weights = ensemble.get_ensemble_prediction(predictions)

# 예측 결과를 데이터프레임에 추가
future_featured['앙상블예측'] = ensemble_pred.astype(int)

# 개별 모델 예측 추가
for name, pred in predictions.items():
    future_featured[f'{name}_예측'] = pred.astype(int)

# 8. 결과 집계 및 저장
def create_forecast_results(future_featured):
    """예측 결과 정리"""
    
    # 일별 데이터 생성 (월별을 일별로 분산)
    daily_results = []
    
    for _, row in future_featured.iterrows():
        month = row['월']
        year = row['연도']
        
        # 해당 월의 일수
        if month in [1, 3, 5, 7, 8, 10, 12]:
            days = 31
        elif month in [4, 6, 9, 11]:
            days = 30
        else:
            days = 28  # 2월 (윤년 무시)
        
        # 월간 예측을 일별로 분산
        daily_avg = row['앙상블예측'] / days
        
        for day in range(1, days + 1):
            # 요일 효과 (주말 +20%, 평일 -5%)
            date_obj = pd.Timestamp(year=year, month=month, day=day)
            weekday = date_obj.weekday()
            
            if weekday >= 5:  # 주말
                daily_visitors = int(daily_avg * 1.2)
            else:  # 평일
                daily_visitors = int(daily_avg * 0.95)
            
            daily_results.append({
                '날짜': date_obj.strftime('%Y-%m-%d'),
                '연도': year,
                '월': month,
                '일': day,
                '요일': weekday,
                '앙상블예측': daily_visitors,
                '국가': row['국적'],
                '목적': row['목적']
            })
    
    # 일별 집계
    daily_df = pd.DataFrame(daily_results)
    daily_summary = daily_df.groupby(['날짜', '연도', '월', '일', '요일']).agg({
        '앙상블예측': 'sum'
    }).reset_index()
    
    # 월별 집계
    monthly_summary = future_featured.groupby('월').agg({
        '앙상블예측': 'sum'
    }).reset_index()
    
    return daily_summary, monthly_summary, future_featured

daily_forecast, monthly_forecast, detailed_forecast = create_forecast_results(future_featured)

# 9. 결과 저장
output_path = 'C:/ai_x/source/proz/머신러닝 학습 분석데이터'
os.makedirs(output_path, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# 파일 저장
daily_path = f"{output_path}/고급ML_일별수요예측_2026_{timestamp}.csv"
monthly_path = f"{output_path}/고급ML_월별수요예측_2026_{timestamp}.csv"
detailed_path = f"{output_path}/고급ML_상세예측_2026_{timestamp}.csv"
performance_path = f"{output_path}/고급ML_모델성능_{timestamp}.csv"

daily_forecast.to_csv(daily_path, index=False, encoding='utf-8-sig')
monthly_forecast.to_csv(monthly_path, index=False, encoding='utf-8-sig')
detailed_forecast.to_csv(detailed_path, index=False, encoding='utf-8-sig')
performance_df.to_csv(performance_path, encoding='utf-8-sig')

print(f"\n💾 예측 결과 저장 완료!")
print(f"📁 일별 예측: {daily_path}")
print(f"📁 월별 예측: {monthly_path}")
print(f"📁 상세 예측: {detailed_path}")
print(f"📁 모델 성능: {performance_path}")

# 10. 최종 결과 요약
print(f"\n" + "="*70)
print("✨ 고급 머신러닝 수요예측 완료! ✨")
print("="*70)

total_2026 = daily_forecast['앙상블예측'].sum()
monthly_avg = monthly_forecast['앙상블예측'].mean()
max_month = monthly_forecast.loc[monthly_forecast['앙상블예측'].idxmax(), '월']
min_month = monthly_forecast.loc[monthly_forecast['앙상블예측'].idxmin(), '월']

print(f"🎯 2026년 총 예측 입국자: {total_2026:,}명")
print(f"📊 월평균 예측: {monthly_avg:,.0f}명")
print(f"📅 일평균 예측: {total_2026/365:,.0f}명")
print(f"🏆 최대 수요월: {max_month}월 ({monthly_forecast.iloc[max_month-1]['앙상블예측']:,}명)")
print(f"📉 최소 수요월: {min_month}월 ({monthly_forecast.iloc[min_month-1]['앙상블예측']:,}명)")

print(f"\n🤖 사용된 모델: {len(ensemble.trained_models)}개")
print(f"🏆 최고 성능 모델: {best_model}")
print(f"📊 최고 정확도: {100-performance_df.loc[best_model, 'MAPE']:.1f}%")
print(f"🎯 특징 수: {len(feature_cols)}개")

print(f"\n🔧 고급 기능:")
print(f"   ✅ 고급 특징 엔지니어링 (시간, 래그, 상호작용, 통계)")
print(f"   ✅ 5개 머신러닝 알고리즘 앙상블")
print(f"   ✅ 자동 특징 선택 및 정규화")
print(f"   ✅ 성능 기반 가중 앙상블")
print(f"   ✅ 현실적 범위 제약 및 검증")

print("="*70)

🚀 고급 머신러닝 수요예측 모델 시작
✅ 데이터 로딩 완료: 59,780개 레코드
📅 기간: 2005년 1월 ~ 2025년 12월
🌍 국가 수: 61개
🎯 목적 종류: 4개
🔧 고급 특징 엔지니어링 시작...
   ✅ 시간 특징 생성 완료
   ✅ 래그 특징 생성 완료
   ✅ 범주형 특징 인코딩 완료
   ✅ 상호작용 특징 생성 완료
   ✅ 통계적 특징 생성 완료
🎯 최종 특징 수: 53개

🎯 고급 머신러닝 모델 학습 시작

🤖 5개 고급 모델 학습 시작...
📊 학습 데이터: 47,824개, 테스트 데이터: 11,956개
   🔄 RandomForest 학습 중...
   ✅ RandomForest: MAPE=8.48%, R²=0.994
   🔄 XGBoost 학습 중...
   ✅ XGBoost: MAPE=83.74%, R²=0.975
   🔄 GradientBoosting 학습 중...
   ✅ GradientBoosting: MAPE=40.37%, R²=0.995
   🔄 Ridge 학습 중...
   ✅ Ridge: MAPE=999.90%, R²=0.967
   🔄 Lasso 학습 중...
   ✅ Lasso: MAPE=999.90%, R²=0.967

📊 모델 성능 상세 분석
------------------------------------------------------------
1. RandomForest    | MAPE:   8.48% | R²:  0.994 | RMSE:      507
2. GradientBoosting | MAPE:  40.37% | R²:  0.995 | RMSE:      450
3. XGBoost         | MAPE:  83.74% | R²:  0.975 | RMSE:     1008
4. Ridge           | MAPE: 999.90% | R²:  0.967 | RMSE:     1157
5. Lasso           | MAPE: 999.90% | R²:  0.967 | RMSE:   